# Exploratory Data Analysis (EDA) on the Titanic Dataset
**Project 01 | Data Science & AI | Tools: Python, Pandas, Seaborn, Matplotlib**

**Goal:** Load the Titanic passenger dataset, clean it, and find out who was most likely to survive.

## Step 1: My three questions
1. **Q1:** Did women survive more than men?
2. **Q2:** Did ticket class matter?
3. **Q3:** Was age related to survival?

I will answer each one with a single sentence containing a real number at the end of the notebook.

## Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

## Step 2: Load and look

In [ ]:
try:
    df = sns.load_dataset("titanic")
except Exception:
    # Fallback: same dataset straight from the seaborn-data repository
    df = pd.read_csv("https://raw.githubusercontent.com/mwaskom/seaborn-data/master/titanic.csv")

print("Shape (rows, columns):", df.shape)

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.describe(include=['object', 'category'])

### What each column means
| Column | Meaning |
|---|---|
| survived | 0 = did not survive, 1 = survived (target) |
| pclass | Ticket class: 1 = upper, 2 = middle, 3 = lower |
| sex | male / female |
| age | Age in years |
| sibsp | Number of siblings or spouses aboard |
| parch | Number of parents or children aboard |
| fare | Ticket price |
| embarked | Port: C = Cherbourg, Q = Queenstown, S = Southampton |
| class | Same as pclass, as text (First, Second, Third) |
| who | man / woman / child |
| adult_male | True if the passenger is an adult male |
| deck | Deck letter of the cabin |
| embark_town | Port name |
| alive | yes / no (same as survived) |
| alone | True if travelling with no family |

## Step 3: Find missing values

In [ ]:
missing = df.isna().sum()
pd.DataFrame({
    "missing": missing,
    "percent": (missing / len(df) * 100).round(1),
})[missing > 0].sort_values("missing", ascending=False)

`deck` is missing for most passengers, `age` for roughly one in five, and `embarked` / `embark_town` for just 2 rows.

## Step 4: Fix them with a reason

| Column | Fix | Why |
|---|---|---|
| `age` | Fill with the **median age of the same class and sex** | Age differs a lot between groups (1st class passengers were older than 3rd class). A group median is more realistic than one overall value, and the median is not distorted by very old or very young passengers. |
| `deck` | **Drop the column** | Most values are missing. Filling that many gaps would mean inventing data. |
| `embarked`, `embark_town` | Fill with the **most common value (mode)** | Only 2 rows are missing, so the most common port is a safe, low-impact choice, and it avoids deleting 2 passengers. |

In [ ]:
# age: median of the same class and sex
df["age"] = df["age"].fillna(df.groupby(["pclass", "sex"])["age"].transform("median"))

# deck: mostly empty, drop it
df = df.drop(columns=["deck"])

# embarked and embark_town: most common value
df["embarked"] = df["embarked"].fillna(df["embarked"].mode()[0])
df["embark_town"] = df["embark_town"].fillna(df["embark_town"].mode()[0])

print("Missing values left:", df.isna().sum().sum())

## Step 5: Check duplicates and types

In [ ]:
dup_count = df.duplicated().sum()
print("Duplicate rows:", dup_count)

**Decision on duplicates:** this dataset has no unique passenger ID column. Two different passengers can share the same class, sex, age, fare, family size and port (for example, a family travelling on the same ticket), so identical-looking rows are not necessarily real duplicates. Dropping them blindly would delete real passengers and change the survival rates, so I **keep them** and note the count above.

In [ ]:
df.dtypes

In [ ]:
# Make categories real categories, and keep numbers as numbers
for col in ["sex", "embarked", "embark_town", "who", "alive"]:
    df[col] = df[col].astype("category")

df["survived"] = df["survived"].astype(int)
df["pclass"] = df["pclass"].astype(int)
df.dtypes

## Step 6: Group and compare

In [ ]:
overall = df["survived"].mean() * 100
print(f"Overall survival rate: {overall:.1f}%")

### By sex

In [ ]:
by_sex = df.groupby("sex", observed=True)["survived"].agg(["mean", "count"]).round(3)
by_sex.sort_values("mean", ascending=False)

### By class

In [ ]:
by_class = df.groupby("pclass")["survived"].agg(["mean", "count"]).round(3)
by_class.sort_values("mean", ascending=False)

### By age band (created with `pd.cut`)

In [ ]:
df["age_band"] = pd.cut(
    df["age"],
    bins=[0, 12, 18, 40, 60, 100],
    labels=["Child", "Teen", "Adult", "Middle-aged", "Senior"],
)
by_age = df.groupby("age_band", observed=True)["survived"].agg(["mean", "count"]).round(3)
by_age.sort_values("mean", ascending=False)

### Extra factors: sex and class together, port, and travelling alone

In [ ]:
df.pivot_table(values="survived", index="sex", columns="pclass", aggfunc="mean", observed=True).round(3)

In [ ]:
df.groupby("embarked", observed=True)["survived"].agg(["mean", "count"]).round(3).sort_values("mean", ascending=False)

In [ ]:
df.groupby("alone")["survived"].agg(["mean", "count"]).round(3).sort_values("mean", ascending=False)

In [ ]:
# Average age and fare per class
df.groupby("pclass").agg(
    passengers=("survived", "count"),
    survival_rate=("survived", "mean"),
    avg_age=("age", "mean"),
    avg_fare=("fare", "mean"),
).round(2)

### Charts

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

sns.barplot(data=df, x="sex", y="survived", ax=axes[0], order=["female", "male"])
axes[0].set_title("Survival rate by sex")

sns.barplot(data=df, x="pclass", y="survived", ax=axes[1])
axes[1].set_title("Survival rate by class")

sns.barplot(data=df, x="age_band", y="survived", ax=axes[2])
axes[2].set_title("Survival rate by age band")

for ax in axes:
    ax.set_ylabel("Survival rate")
plt.tight_layout()
plt.show()

In [ ]:
sns.barplot(data=df, x="pclass", y="survived", hue="sex")
plt.title("Survival rate by class and sex")
plt.ylabel("Survival rate")
plt.show()

## Step 7: Answer my questions
Each sentence below is generated from the data, so the numbers are always the real results.

In [ ]:
# Q1: sex
sex_rate = (df.groupby("sex", observed=True)["survived"].mean() * 100).round(1).sort_values(ascending=False)
top_sex, low_sex = sex_rate.index[0], sex_rate.index[-1]
q1 = (f"Q1: {top_sex.capitalize()} survived at about {sex_rate.iloc[0]:.0f} percent "
      f"compared with about {sex_rate.iloc[-1]:.0f} percent for {low_sex}.")

# Q2: class
cls = (df.groupby("pclass")["survived"].mean() * 100).round(1)
q2 = (f"Q2: Yes, class mattered: survival fell from about {cls[1]:.0f} percent in 1st class "
      f"to {cls[2]:.0f} percent in 2nd class and {cls[3]:.0f} percent in 3rd class.")

# Q3: age
age_rate = (df.groupby("age_band", observed=True)["survived"].mean() * 100).round(1)
best, worst = age_rate.idxmax(), age_rate.idxmin()
q3 = (f"Q3: Age was related to survival: the '{best}' band survived most at about {age_rate.max():.0f} percent, "
      f"while the '{worst}' band survived least at about {age_rate.min():.0f} percent.")

for q in (q1, q2, q3):
    print(q, "\n")

## Summary of my three findings
*(Run the cell above. These are the same results in written form.)*

1. **Sex:** Women had a far higher survival rate than men. This fits the "women and children first" evacuation rule.
2. **Class:** Survival dropped sharply from 1st to 3rd class. Higher classes also paid higher fares and were housed closer to the lifeboats.
3. **Age:** Children had the best survival odds of any age band, while the lowest survival was in the older adult bands.

**Careful wording:** these are *associations*, not proof of cause. Sex, class and age are also linked to each other (for example, many 3rd class passengers were adult men), so one factor can partly explain another. Cleaning choices (median age fill, dropping `deck`, keeping possible duplicates) could shift the exact numbers slightly.

## Conclusion
Using Pandas I inspected the data (`shape`, `head`, `info`, `describe`), handled missing values with a stated reason for each fix, checked duplicates and data types, and used `groupby`, `agg` and `pivot_table` to compare survival across sex, class, age band, port and travelling alone. Survival depended strongly on **sex, class and age**.